# 03 · Funnel Analysis

Digital Onboarding Optimization Case Study

> Fully synthetic dataset. No real company, customer, or proprietary information is included.

## 1. Objective

This notebook converts the event and user-level datasets into a product analytics funnel.

The goal is to quantify where users drop off and how document-upload timing changes the probability of reaching approval.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

users = pd.read_csv(DATA_DIR / "onboarding_users.csv", parse_dates=["signup_timestamp"])
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_timestamp"])
support = pd.read_csv(DATA_DIR / "support_calls.csv", parse_dates=["call_timestamp"])

users.head()

## 2. Funnel Definition

In [ ]:
funnel_steps = [
    ("app_opened", "App Opened"),
    ("onboarding_started", "Onboarding Started"),
    ("personal_data_completed", "Personal Data Completed"),
    ("document_screen_viewed", "Document Screen Viewed"),
    ("document_submitted", "Document Submitted"),
    ("identity_approved", "Identity Approved")
]

step_counts = []
for event_name, label in funnel_steps:
    step_counts.append({
        "step": label,
        "event_name": event_name,
        "users": events.loc[events["event_name"].eq(event_name), "user_id"].nunique()
    })

funnel = pd.DataFrame(step_counts)
funnel["conversion_from_start"] = funnel["users"] / funnel.loc[0, "users"]
funnel["step_conversion"] = funnel["users"] / funnel["users"].shift(1)
funnel.loc[0, "step_conversion"] = 1
funnel

In [ ]:
ax = funnel.set_index("step")["users"].plot(kind="bar", figsize=(11, 5))
ax.set_title("Onboarding Funnel - Unique Users by Step")
ax.set_ylabel("Unique users")
ax.set_xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 3. Funnel by Upload Choice

In [ ]:
funnel_by_choice = users.groupby("upload_choice").agg(
    users=("user_id", "count"),
    p1_completed=("p1_completed", "mean"),
    p2_reached=("p2_reached", "mean"),
    document_submitted=("document_submitted", "mean"),
    approved=("approved", "mean")
)
funnel_by_choice

In [ ]:
plot_df = funnel_by_choice[["p1_completed", "p2_reached", "document_submitted", "approved"]].T
ax = plot_df.plot(figsize=(10, 5), marker="o")
ax.set_title("Funnel Rates by Upload Choice")
ax.set_ylabel("Rate")
ax.set_xlabel("Funnel Step")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 4. Drop-off Analysis

In [ ]:
dropoff = funnel.copy()
dropoff["users_lost_vs_previous_step"] = dropoff["users"].shift(1) - dropoff["users"]
dropoff["dropoff_rate_vs_previous_step"] = 1 - dropoff["step_conversion"]
dropoff

## 5. Segment Funnel Heatmap

In [ ]:
segment = users.groupby("channel").agg(
    p1_completed=("p1_completed", "mean"),
    p2_reached=("p2_reached", "mean"),
    document_submitted=("document_submitted", "mean"),
    approved=("approved", "mean"),
    upload_now_rate=("upload_choice", lambda s: (s == "upload_now").mean())
).sort_values("approved", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(segment)
ax.set_xticks(range(len(segment.columns)))
ax.set_yticks(range(len(segment.index)))
ax.set_xticklabels(segment.columns, rotation=45, ha="right")
ax.set_yticklabels(segment.index)
ax.set_title("Funnel KPI Heatmap by Acquisition Channel")
fig.colorbar(im)
plt.tight_layout()
plt.show()

segment

## 6. Business Interpretation

The funnel confirms the central behavioral pattern:

> The document-submission step is the main bottleneck, and users who choose to upload later are far less likely to recover and reach approval.

This supports prioritizing UX interventions around the document screen.